In [3]:
from DeviceClasses.E5063A_VNA_Class import VNA
from DeviceClasses.RS_FSW_new import FSW
from DeviceClasses.SMA100B import Signal_Generator
from DeviceClasses.Utilities.Fitting import fit_S21_peak, S21_eq, linear_voltage_to_dB
from datetime import datetime
import matplotlib.pyplot as plt
%matplotlib qt

In [ ]:

FSW_address = "TCPIP::134.219.128.35::INSTR"
VNA_address = "TCPIP::134.219.128.52::INSTR"
SG_address  = 'TCPIP::134.219.128.14::INSTR'

#plots:
fig_SA, ax_SA = plt.subplots()
ax_SA.set(title = "Spectrum", xlabel = "frequency[Hz]", ylabel = "power/drive_power[dBm]")
   
#general parameters:
base_path = r"C:\Experiments\Bulk_He_2\16122024\\"


## FSW parameters:
central_freq = 6188462993.528336
span = 100e3
sweep_points = 100001
band_width = 1
video_band_width = band_width
average_number = 50
sweep_time  = 0.01
ref_level = -100
detector_type = "SAMPLE"
ref = "EXT"
save = True
file_location = base_path
file_name = f""
extra_meta = {}


FSW_sweep_parameter_dict = {
"central_freq": central_freq, 
"span":span, 
"band_width":band_width,
"sweep_points":sweep_points, 
"video_band_width":video_band_width,
"average_number":average_number,
"sweep_time":sweep_time,
"ref_level":ref_level,
"detector_type":detector_type,
"ref":ref,
"save":save, 
"file_location":file_location, 
"file_name":file_name,
"extra_meta":extra_meta
}


## VNA parameters
channels = [1]      # formatting: [1,2,3,4]
measuring = ["S21"] # formatting: ["S12", "S21"..]
traces = ["1"] # formatting: ["CH1TR1", "CH2TR1"]
cent_freq=6188249936
span=4e6
freq_start = cent_freq-span/2
freq_stop = cent_freq+span/2
number_of_point=1001
band_width = 1e3
power=0

#number of S21 sweeps
S21_checks = 5


##SG parameters


## Sweep parameters:
sweep_powers = range(0, 16)

#extra_parameters:
Temp = 30e-3



with Signal_Generator(SG_address) as SG:
    with VNA(VNA_address,) as vna:
        with FSW(FSW_address, timeout = None) as SA:

            SG.turn_mw("On")
            #sweep for loops
            for drive_power in sweep_powers:
                


                #VNA Sweep
                fig_S21, ax_S21 = plt.subplots()
                ax_S21.set(title = f"S21 data and Fit", ylabel = "|S21| [dB]", xlabel = "frequency[Hz]")

                for channel, matrix, trace_name in zip(channels,\
                                                measuring,\
                                                traces):
                    for S21_check in range(S21_checks):
                        
                        #sweep
                        timestamp_S21 = datetime.now().strftime("%Y_%m_%d_%H_%M_%S_%f")
                        file_name_prefix = f'{timestamp_S21}_S21check_{S21_check}_Temp_{Temp}_K_drivepowerdBm_{drive_power}.tsv'
                        print("useful:",channel, matrix)
                        data = vna.sweep(channel, matrix, trace_name, freq_start, freq_stop, number_of_point,\
                                        power, band_width,\
                                        _file_name=file_name_prefix, file_location= base_path, reset = True)
                        amp_S21 = data[0]
                        frequency_S21 = data[2]

                        #fitting
                        signal_applied = [band_width, 1]
                        popt, pcov, guesses, amplitude_linear_voltage, frequencies = fit_S21_peak(amp_S21, frequency_S21, guesses = False, signal_applied = signal_applied)
                        
                        #Plot the data
                        ax_S21.plot(frequency_S21, amp_S21, 'C0-', label='Data (S21 in Power)')
                        ax_S21.plot(frequency_S21, linear_voltage_to_dB(S21_eq(frequency_S21, *popt)), 'k--', label=f'Fit in Power Domain {S21_check}, popt:{popt}')
                        ax_S21.legend()
                        fig_S21.show()
                        resonance_freq=float(popt[0])
                        drive_freq = resonance_freq
                        print(f"Estimated Resonance Frequency: {popt[0]} GHz")
                        
                        #update signal generator settings 
                        SG.set_mw_frequency(drive_freq)
                        SG.set_mw_power(drive_power)
                        SG.turn_mw('ON')
                        
                        #time between sweeps
                        plt.pause(5)

                        SA._device.timeout = None



                
                FSW_sweep_parameter_dict["extra_meta"] =  FSW_sweep_parameter_dict["extra_meta"] | {"drive_power_dBm":drive_power,
                                                                                                    "drive_freq_Hz"  :drive_freq,
                                                                                                    "temperature_K"  :Temp
                                                                                                    }
                
                FSW_sweep_parameter_dict["central_freq"]  =  drive_freq + 49e3

                timestamp = datetime.now().strftime("%Y_%m_%d_%H_%M_%S_%f")
                FSW_sweep_parameter_dict["file_name"]  =  f"{timestamp}_SA_drivepower_{drive_power}.tsv"
                powers, frequencies = SA.sweep(**FSW_sweep_parameter_dict)

                print(powers)
                print(frequencies)
                ax_SA.plot(frequencies- drive_freq, powers-drive_power, label = f"drive power:{drive_power}")
                fig_SA.show()

                ax_SA.legend()


            #always turn off!
            SG.turn_mw("Off")
                    

useful: 1 S21
error info pre-run check 
+0,"No error"

error info 321
+0,"No error"

error info 328
+0,"No error"

error info 342
+0,"No error"

error info 345
+0,"No error"

turning power on
sweep time: 1

expected time: +1.93152100000E+000

error info 357
+0,"No error"

sweep running, dont turn off
error info 361
+0,"No error"

read: b'+1\n'
Sweep finished, saving
error info 379: 1, 1
 +0,"No error"


              data1 = [-76.51110534 -77.09624466 -75.98813683 ... -83.20492961 -78.54930697
 -82.09703424]
              data1shape = (1001,)
              
              data2 = [ 72.3296428   82.65854804  69.95421288 ... 152.93718606 135.60340868
 159.36353455] 
              data1shape = (1001,)
              
              data_freq = [6.18624994e+09 6.18625394e+09 6.18625794e+09 ... 6.19024194e+09
 6.19024594e+09 6.19024994e+09]
              data_freq = (1001,)
              
saving as: C:\Experiments\Bulk_He_2\16122024\\2024_12_16_12_42_30_295747_S21check_0_Temp_0.03_K_drivepower